# Formatos, importação e extração

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_03/01_formatos_importacao_e_extracao.ipynb)

O guia separou dados brutos, intermediários e derivados. Este notebook começa
pelo ponto de entrada desse fluxo: reconhecer o que cada arquivo contém e
escolher uma operação de leitura que preserve sua estrutura e proveniência.

## Linha do tempo: formatos surgem para problemas diferentes

Formatos não aparecem todos no mesmo contexto nem possuem necessariamente uma
única “data de criação”. Em alguns casos conhecemos o início do projeto; em
outros, o primeiro rascunho público ou a data de uma padronização posterior.
A linha do tempo usa o marco mais bem documentado e identifica sua natureza.

![Linha do tempo mostra HTML em 1990, XML entre 1996 e 1998, JSON e YAML em 2001, a documentação do CSV em 2005 e a padronização do XLSX em 2006.](imagens/01_linha_tempo_formatos.svg)

| Marco | Formato | Objetivo inicial ou problema enfrentado | Natureza da data |
|---|---|---|---|
| 1990 | HTML | publicar documentos com ligações no projeto World Wide Web | criação no projeto Web |
| 1996 → 1998 | XML | oferecer uma forma simplificada de SGML para informação estruturada e intercâmbio na Web | primeiro *Working Draft* → Recomendação W3C |
| 2001 | JSON | representar objetos e listas em uma sintaxe textual pequena e independente de linguagem para intercâmbio | apresentação pública no JSON.org |
| 2001 | YAML | serializar estruturas de dados com prioridade para legibilidade humana e interação com linguagens de programação | primeiros rascunhos públicos |
| 2005 | CSV | registrar convenções comuns para intercambiar tabelas planas em texto | RFC 4180; documentação de uso anterior, não criação |
| 2006 | XLSX / Office Open XML | representar e empacotar documentos de escritório, incluindo planilhas, com vocabulários baseados em XML | primeira edição da ECMA-376 |

A posição do CSV exige atenção: a RFC 4180 afirma que o formato já era usado
havia bastante tempo. Portanto, 2005 é a data de sua documentação em RFC, não
um “aniversário” do CSV. A cronologia também corrige outra simplificação comum:
XLSX não foi criado em 2001; o marco normativo adotado aqui é a primeira edição
da ECMA-376, publicada em 2006.

HTML e YAML entram como contexto comparativo, embora não sejam manipulados nos
exercícios deste notebook. O percurso prático se concentra em CSV, XLSX, JSON
e XML, além de TXT e PDF, porque são os formatos presentes na pasta de dados.
A história ajuda a perceber que a escolha entre eles não é uma escala do
“mais antigo” ao “mais moderno”: cada formato favorece objetivos distintos.
Os documentos históricos e as especificações usados para estabelecer os
marcos — W3C, IETF/RFC, ECMA e YAML — estão listados em `referencias.md`.

## 1. Formato é estrutura e *affordance* (possibilidades de ação)

Aqui, *affordance* designa as ações que um formato facilita, dificulta ou
condiciona para quem o utiliza.

CSV registra uma tabela sem fórmulas ou tipos ricos; XLSX pode conter várias
planilhas, fórmulas e formatação; JSON e XML expressam hierarquias; TXT não
define internamente como interpretar seu conteúdo; PDF busca preservar uma
apresentação de página e pode conter texto, imagem ou ambos. Extensão não
garante conteúdo nem qualidade.

![Quatro painéis comparam CSV como grade tabular, XLSX como pasta com planilhas, JSON como árvore de objetos e listas e XML como árvore de elementos aninhados.](imagens/01_formatos_estruturas.svg)

A figura destaca a estrutura predominante e as operações favorecidas por cada
formato. CSV e XLSX podem representar tabelas, mas uma pasta XLSX pode reunir
abas, fórmulas e formatação. JSON e XML preservam hierarquias, embora usem
sintaxes e formas de navegação diferentes. Em todos os casos, a extensão
orienta a escolha inicial do leitor, mas parâmetros e testes confirmam se a
interpretação foi correta.

In [1]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_03'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('openpyxl', 'openpyxl>=3.1,<4'), ('pypdf', 'pypdf>=5,<6'), ('PIL', 'Pillow>=11,<12')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Ambiente local: nenhuma clonagem necessária.


In [2]:
import json
from pathlib import Path
from xml.etree import ElementTree as ET
import pandas as pd

pasta = Path("dados/brutos")
csv_df = pd.read_csv(pasta / "catalogo_messy.csv", sep=";")
xlsx_df = pd.read_excel(pasta / "catalogo_messy.xlsx", sheet_name="documentos")
json_data = json.loads((pasta / "metadados.json").read_text(encoding="utf-8"))
xml_raiz = ET.parse(pasta / "metadados.xml").getroot()
txt = (pasta / "D001.txt").read_text(encoding="utf-8")

print("CSV/XLSX iguais nas dimensões:", csv_df.shape == xlsx_df.shape)
print("Registros JSON:", len(json_data), "| XML:", len(xml_raiz))
print("TXT:", txt.strip())

CSV/XLSX iguais nas dimensões: True
Registros JSON: 3 | XML: 2
TXT: A escola noturna foi apresentada como sinal de progresso.


As saídas confirmam que os leitores devolveram objetos utilizáveis, mas esse
sucesso técnico ainda não informa se a estrutura foi compreendida corretamente.
A interpretação começa comparando o resultado com aquilo que esperávamos ler.

### Interpretação

Leitores diferentes retornam objetos diferentes. Importar com sucesso não
prova que encoding, separador, planilha, hierarquia ou tipos foram interpretados
corretamente. Registre parâmetros, versão da fonte e testes esperados.

A seção anterior tratou da **estrutura interna** dos arquivos: leitor,
separador, planilha, hierarquia e tipos. Quando os dados vêm de uma instituição
externa, surge uma segunda exigência: documentar **qual versão** foi utilizada,
de onde veio, quando foi acessada e qual recorte foi preservado.

## 2. Base pública: cópia local, versão e proveniência

O arquivo `extrato_codigos_municipios_ibge.csv` é uma cópia local pequena da
tabela do IBGE. Ele permite repetir o exercício em sala sem depender da rede e
mantém estável a entrada usada pelo código. Essa conveniência técnica não torna
a cópia autossuficiente: uma base pública pode mudar, e citar apenas “IBGE” não
identifica a versão efetivamente analisada.

Por isso, o CSV é acompanhado por `proveniencia_base_publica.json`, que registra
fonte, endereço, data de acesso, recorte e ressalva de reutilização. Os dois
arquivos respondem a perguntas complementares:

- o CSV responde **quais registros e campos foram preservados?**;
- o JSON responde **de onde veio esta cópia e em que condições ela pode ser
  compreendida e atualizada?**

No código seguinte, `dtype={"codigo_municipio": "string"}` também explicita uma
decisão semântica: o código municipal é um identificador, não uma quantidade
sobre a qual se devam realizar operações aritméticas.

In [3]:
municipios = pd.read_csv(
    pasta / "extrato_codigos_municipios_ibge.csv",
    dtype={"codigo_municipio": "string"},
)
proveniencia = json.loads(
    (pasta / "proveniencia_base_publica.json").read_text(encoding="utf-8")
)
print(proveniencia["fonte"], "— acesso:", proveniencia["data_de_acesso"])
municipios

IBGE — Códigos dos Municípios — acesso: 2026-07-31


,codigo_municipio,nome_municipio_ibge,uf
0,3550308,São Paulo,SP
1,3106200,Belo Horizonte,MG
2,3304557,Rio de Janeiro,RJ
3,2611606,Recife,PE


Com o CSV, tratamos uma fonte externa já organizada como tabela: foi necessário
interpretar sua estrutura e documentar sua versão. Agora mudaremos de tipo de
objeto. Documentos orientados à página não oferecem diretamente linhas e
colunas; antes de escolher uma técnica, precisamos descobrir se o PDF contém
caracteres codificados ou apenas imagem.

## 3. Extração de PDF não é OCR

Se o PDF contém caracteres, um leitor pode extrair a camada textual. Se cada
página é apenas imagem, a extração retorna pouco ou nada e será necessário OCR.
Mesmo PDF com texto pode ter ordem de leitura problemática, hifenização ou
caracteres incorretos.

![Um arquivo PDF passa pelo teste de texto selecionável: quando há texto, extrai-se a camada textual; quando não há, aplica-se OCR; as duas rotas exigem avaliação contra a página.](imagens/01_pdf_texto_imagem_ocr.svg)

O diagrama evita uma confusão frequente: OCR não é sinônimo de “abrir PDF”.
Primeiro se diagnostica o conteúdo; depois se escolhe e avalia a operação.

In [4]:
from pypdf import PdfReader

leitor = PdfReader(pasta / "documento_textual.pdf")
texto_pdf = "\n".join(pagina.extract_text() or "" for pagina in leitor.pages)
print(texto_pdf.strip())

Documento D001 com camada textual


A extração anterior acessou uma camada textual já presente no PDF. Quando essa
camada não existe ou é inadequada, o texto precisa ser estimado a partir dos
pixels; por isso, o OCR deve ser tratado como hipótese de transcrição.

## 4. OCR como hipótese de transcrição

OCR reconhece caracteres em imagem. Resolução, inclinação, ruído, layout,
fonte e idioma afetam o resultado. A documentação do Tesseract recomenda
inspecionar e preparar imagens quando necessário. A transcrição deve ser
vinculada à imagem, à ferramenta, aos parâmetros e a uma avaliação de erro.

As duas imagens abaixo são **simulações didáticas**, não documentos
históricos. Elas contêm a mesma linha: a primeira está limpa; a segunda foi
reduzida, ampliada, inclinada, desfocada e marcada de forma controlada.

![Linha tipográfica sintética limpa usada como referência no experimento controlado de reconhecimento óptico de caracteres.](dados/brutos/pagina_digitalizada.png)

![A mesma linha tipográfica sintética após redução, ampliação, inclinação, desfoque e marcas controladas que dificultam o reconhecimento óptico de caracteres.](dados/brutos/pagina_digitalizada_degradada.png)

In [5]:
import shutil
import subprocess

entradas_ocr = [
    ("imagem limpa", pasta / "pagina_digitalizada.png", pasta / "ocr_precomputado.txt"),
    ("imagem degradada", pasta / "pagina_digitalizada_degradada.png", pasta / "ocr_precomputado_degradado.txt"),
]

def reconhecer_ou_carregar(imagem, precomputado):
    if shutil.which("tesseract"):
        try:
            processo = subprocess.run(
                ["tesseract", str(imagem), "stdout", "-l", "eng", "--psm", "7"],
                capture_output=True, text=True, check=True,
            )
            return processo.stdout.strip(), "Tesseract executado agora"
        except subprocess.CalledProcessError:
            pass
    return (
        precomputado.read_text(encoding="utf-8").strip(),
        "saída pré-computada fornecida com o material",
    )

resultados_ocr = []
for condicao, imagem, precomputado in entradas_ocr:
    transcricao, origem = reconhecer_ou_carregar(imagem, precomputado)
    resultados_ocr.append({"condição": condicao, "transcrição": transcricao, "origem": origem})

pd.DataFrame(resultados_ocr)

,condição,transcrição,origem
0,imagem limpa,ESCOLA NOTURNA E TRABALHO — 1890,Tesseract executado agora
1,imagem degradada,ESCOLA NOTURNA E TRABALHO -— 1890,Tesseract executado agora


In [6]:
referencia = (pasta / "ocr_referencia.txt").read_text(encoding="utf-8").strip()

def distancia_edicao(a, b):
    anterior = list(range(len(b) + 1))
    for i, item_a in enumerate(a, 1):
        atual = [i]
        for j, item_b in enumerate(b, 1):
            atual.append(min(
                atual[-1] + 1,
                anterior[j] + 1,
                anterior[j - 1] + (item_a != item_b),
            ))
        anterior = atual
    return anterior[-1]

comparacao = []
for resultado in resultados_ocr:
    observado = resultado["transcrição"]
    comparacao.append({
        "condição": resultado["condição"],
        "erros em caracteres": distancia_edicao(referencia, observado),
        "CER": distancia_edicao(referencia, observado) / len(referencia),
        "erros em palavras": distancia_edicao(referencia.split(), observado.split()),
        "WER": distancia_edicao(referencia.split(), observado.split()) / len(referencia.split()),
    })

pd.DataFrame(comparacao).round({"CER": 3, "WER": 3})

,condição,erros em caracteres,CER,erros em palavras,WER
0,imagem limpa,0,0.000,0,0.000
1,imagem degradada,1,0.031,1,0.167


### Testes de importação do arquivo do projeto

Antes do inventário, verifico como o CSV do Goodreads reage a parâmetros
corretos e incorretos de leitura.

In [7]:
# Localiza a pasta do projeto Goodreads (localmente ou no Colab, se a pasta for enviada para /content)
import sys
from pathlib import Path

CANDIDATOS = [Path("../projeto_goodreads"), Path("projeto_goodreads"), Path("/content/projeto_goodreads")]
PROJETO = next((c.resolve() for c in CANDIDATOS if (c / "preparar_base.py").exists()), None)
if PROJETO is None:
    raise FileNotFoundError(
        "Pasta projeto_goodreads não encontrada. Coloque-a ao lado das pastas unidade_0X "
        "ou, no Colab, envie-a para /content."
    )
if str(PROJETO) not in sys.path:
    sys.path.insert(0, str(PROJETO))
print("Projeto localizado:", (PROJETO / "preparar_base.py").exists())

from preparar_base import BRUTO, SHA256_ESPERADO, sha256_arquivo, COLUNAS_ESPERADAS

# Leitura correta: UTF-8 com BOM e vírgula como separador
goodreads = pd.read_csv(BRUTO, sep=",", encoding="utf-8-sig")
print("Dimensões:", goodreads.shape)
print("Colunas conforme o esperado:", list(goodreads.columns) == COLUNAS_ESPERADAS)
print("id_livro único:", goodreads["id_livro"].is_unique)
print("Hash confere:", sha256_arquivo(BRUTO) == SHA256_ESPERADO)

# Leitura com codificação errada: o arquivo abre, mas o conteúdo fica corrompido
errado = pd.read_csv(BRUTO, sep=",", encoding="latin-1")
print("\nPrimeira coluna lida com latin-1:", repr(errado.columns[0]))
print("Exemplos corrompidos:", errado.loc[errado["titulo"].str.contains("Ã", regex=False), "titulo"].head(3).tolist())

# Leitura com separador errado: falha explícita
try:
    pd.read_csv(BRUTO, sep=";", encoding="utf-8-sig")
except Exception as erro:
    print("\nSeparador ';' falha:", type(erro).__name__)

Projeto localizado: True
Dimensões: (10454, 15)
Colunas conforme o esperado: True
id_livro único: True
Hash confere: True

Primeira coluna lida com latin-1: 'ï»¿id_livro'
Exemplos corrompidos: ['HÃ´tel Transylvania', 'Pedro PÃ¡ramo', 'My Ã\x81ntonia']

Separador ';' falha: ParserError


As duas transcrições e suas métricas permitem comparar condições controladas.
Antes de decidir se uma saída é aceitável, precisamos relacionar a magnitude
do erro à finalidade concreta para a qual o texto será utilizado.

### Como interpretar a comparação

CER é a taxa de erro por caractere; WER, por palavra. Zero significa
coincidência com a transcrição de referência nesta amostra. Valores maiores
indicam mais edições necessárias, mas **não decidem sozinhos** se o texto é
adequado: busca exploratória, citação e análise lexical exigem tolerâncias
diferentes. Uma referência também pode conter erro humano; por isso, a
página e o protocolo de transcrição continuam indispensáveis.

A comparação de CER e WER transforma diferenças visíveis em indicadores de
qualidade, mas a adequação continua dependente do uso pretendido. O inventário
técnico reúne formato, operação, parâmetro, teste e finalidade para todas as
fontes do projeto.

## U03-A02 — Atividade — inventário técnico

Para cada fonte do projeto, registre formato, estrutura interna, leitor,
encoding/planilha/nó, presença de texto, necessidade de OCR, riscos, teste e
saída prevista. Diferencie claramente dado recebido, texto extraído e texto
reconhecido.

**Inventário:** (os testes estão na célula de código imediatamente anterior)

| Fonte | Formato e estrutura | Leitor e parâmetros | Texto e OCR | Riscos | Teste | Saída prevista |
|---|---|---|---|---|---|---|
| `livros_good_reads.csv` | CSV tabular, 10.454 linhas e 15 colunas, uma linha por livro | `pd.read_csv(sep=",", encoding="utf-8-sig")` | **dado recebido**: títulos e nomes já são texto digital; não há PDF nem imagem; OCR desnecessário | ler com `latin-1` corrompe acentos e alfabetos não latinos (ex.: “Pedro PÃ¡ramo”) e deixa o BOM grudado no nome da primeira coluna; separador errado quebra a leitura | número de linhas e colunas, nomes das colunas, unicidade de `id_livro`, hash SHA-256 | `dados/intermediarios/livros_normalizado.csv` |
| `proveniencia_goodreads.json` | JSON com chaves e listas | `json.loads(...read_text(encoding="utf-8"))` | não se aplica | omissões não aparecem como erro de leitura | presença das chaves e conferência do hash | documentação do pacote |
| `dicionario_dados.csv` | CSV tabular, um campo por linha | `pd.read_csv` | não se aplica | dicionário desatualizado em relação aos dados | comparação entre campos documentados e colunas observadas | documentação do pacote |
| Páginas do Goodreads (consulta pontual de casos) | HTML | navegador, consulta manual | **texto lido por pessoa**, não extraído | página pode ter mudado desde a coleta | registrar data de consulta | anotações de casos |

**Distinção entre camadas de texto:** no projeto só há **dado recebido** (texto
digital que veio no CSV). Não há **texto extraído** de PDF nem **texto
reconhecido** por OCR. Se, no futuro, eu coletar resenhas, elas também serão
texto digital extraído de HTML, não OCR, e exigirão registro de data e versão
da página.

Ao preencher o inventário, o estudante torna explícito como cada entrada será
lida e avaliada. A síntese fixa o princípio que deverá acompanhar a próxima
etapa, quando essas representações serão reorganizadas e normalizadas.

## Síntese

A importação é interpretação técnica. O objetivo não é converter tudo para um
único formato sem crítica, mas produzir representações adequadas, vinculadas e
testáveis. Leve o inventário ao Notebook 02: ele orientará a unidade da linha,
as regras de transformação e os testes de qualidade.